### 离散数值迭代的非线性带不等式约束的LQR解法

在实际应用中，非线性系统的LQR问题通常需要通过数值迭代的方法求解，尤其是当系统受到不等式约束时。以下是基于前向-后向迭代的数值解法的详细描述：

#### 问题描述
我们需要求解以下优化问题：
$$
\min \quad J = \sum_{k=0}^{N-1} L(x_k, u_k) + \Phi(x_N),
$$
约束条件为：
1. **动态约束**：
$$
x_{k+1} =f(x_k, u_k), 
$$
2. **不等式约束**：
$$
g(x_k, u_k) \leq 0.
$$

#### 哈密顿函数构造
为了处理上述优化问题，我们引入协态变量 $\lambda_k$ 和拉格朗日乘子 $\mu_k$，构造离散形式的哈密顿函数：
$$
H(x_k, u_k, \lambda_{k+1}, \mu_k) = L(x_k, u_k) + \lambda_{k+1}^T f(x_k, u_k) + \mu_k^T g(x_k, u_k),
$$
其中 $\mu_k \geq 0$ 是与不等式约束相关的乘子。

#### 哈密顿函数的极值条件

在离散时间系统中，连续的微分方程被差分方程替代，协态方程的方向和形式有所不同。以下是离散情况下的相应条件：

1. **状态方程（动力学约束）**：
   $$
   x_{k+1} =f(x_k, u_k),
   $$
   （对应连续的 $\dot{x} = f$，通过离散化得到。）

2. **协态方程（伴随方程）**：
   $$
   \lambda_k =\frac{\partial L}{\partial x_k} + \mu_k^T \frac{\partial g}{\partial x_k} + \lambda_{k+1}^T \frac{\partial f}{\partial x_k} ,
   $$

3. **控制最优性条件**：
   $$
   \frac{\partial H_k}{\partial u_k}=\frac{\partial L}{\partial u_k} + \lambda_{k+1}^T \frac{\partial f}{\partial u_k} + \mu_k^T \frac{\partial g}{\partial u_k} = 0,
   $$
   （对应连续的 $\frac{\partial H}{\partial u} = 0$，形式相同。）

4. **互补条件**：
   $$
   \mu_k^T g(x_k, u_k) = 0, \quad \mu_k \geq 0, \quad g(x_k, u_k) \leq 0.
   $$
   （与连续情况相同。）

#### 边界条件
1. 初始状态：
   $$
   x_0 \text{ 给定},
   $$
2. 终端条件：
   $$
   \lambda_N = \frac{\partial \Phi}{\partial x_N}.
   $$

离散情况通过数值迭代（如前向-后向方法）求解，而连续情况通常通过解析或数值积分求解。

#### 数值迭代方法(梯度下降)
数值解法基于前向-后向迭代，具体步骤如下：

1. **初始化**：
   - 设置初始状态 $x_0$。
   - 初始化控制序列 $u_k$（例如，零控制或随机初始化）。
   - 初始化拉格朗日乘子 $\mu_k$（如果有不等式约束）。

2. **前向积分**：
   - 从 $x_0$ 出发，利用当前的 $u_k$，通过动态方程计算状态序列 $x_k$：
     $$
     x_{k+1} = f(x_k, u_k).
     $$

3. **后向积分**：
   - 从终端条件 $\lambda_N = \frac{\partial \Phi}{\partial x_N}$ 出发，利用当前的 $x_k$ 和 $u_k$，通过协态方程计算协态序列 $\lambda_k$：
     $$
     \lambda_k = \frac{\partial L}{\partial x_k} + \lambda_{k+1}^T \frac{\partial f}{\partial x_k} + \mu_k^T \frac{\partial g}{\partial x_k}.
     $$
     注意，这里需要使用前向积分得到的 $x_k$ 和 $u_k$。

4. **更新控制变量**：
   - 对每个时间步 $k$，根据控制最优性条件更新 $u_k$：
     $$
     \frac{\partial H}{\partial u_k} = \frac{\partial L}{\partial u_k} + \lambda_{k+1}^T \frac{\partial f}{\partial u_k} + \mu_k^T \frac{\partial g}{\partial u_k} = 0.
     $$
   - 如果 $\frac{\partial H}{\partial u_k} = 0$ 是线性方程，可以直接求解 $u_k$。
   - 如果是非线性方程，使用梯度下降法或牛顿法迭代求解：
     $$
     u_k^{(i+1)} = u_k^{(i)} - \eta \cdot \frac{\partial H}{\partial u_k}.
     $$
   - 如果有不等式约束 $g(x_k, u_k) \leq 0$，将 $u_k$ 投影到可行域内：
     $$
     u_k = \arg\min_{u} \| u - u_k \|^2, \quad \text{subject to } g(x_k, u) \leq 0.
     $$

5. **更新拉格朗日乘子**：
   - 在算完当前步的最优控制量 $u_k^*$ 之后，使用当前 $x_k$ 一起更新乘子。
   - 根据互补条件更新 $\mu_k$：
     $$
     \mu_k = \max(0, \mu_k + \alpha \cdot g(x_k, u_k)).
     $$

6. **收敛判定**：
   - 定义收敛准则，例如：
     $$
     \| u_k^{(i+1)} - u_k^{(i)} \| < \epsilon.
     $$
   - 这一步需要在当前循环结束后，对所有时间步都检查是否满足。
   - 如果满足收敛条件，则停止迭代；否则，返回前向积分步骤。

#### 总结
上述方法通过前向-后向迭代，逐步逼近最优解。每次迭代中，状态变量 $x_k$、控制变量 $u_k$ 和协态变量 $\lambda_k$ 都会被更新。不等式约束通过投影法或拉格朗日乘子法处理，确保解的可行性。

### 回顾ILQR核心递推过程
**二阶展开（在标称轨迹点）**：

设标称点 $(x_k,u_k)$，令增量 $d x,d u$，用线性化动力学
$$
d x_{k+1}\approx A\,d x + B\,d u,\quad A=\partial x_{k+1}/\partial x,\;B=\partial x_{k+1}/\partial u.
$$
对 $\ell$ 做二阶展开
$$
\ell(x+d x, u+d u) \approx \ell + \ell_x^Td x + \ell_u^Td u + \tfrac12 [d x \quad d u]^T 
\begin{bmatrix}\ell_{xx} & \ell_{xu} \\ \ell_{ux} & \ell_{uu} \end{bmatrix} [d x \quad d u]
$$
对$V_{k+1}$ 做二阶展开
$$
V_{k+1}(x_{k+1}+d x_{k+1}) \approx V_{k+1} + V_{k+1,x}^Td x_{k+1} + \tfrac12\,d x_{k+1}^T V_{k+1,xx}\,d x_{k+1}.
$$
组合Q函数：
$$
\begin{aligned}
Q(x+dx,u+du) &\approx \ell + \ell_x^Td x + \ell_u^Td u + \tfrac12 [d x \quad d u]^T \begin{bmatrix}\ell_{xx} & \ell_{xu} \\
\ell_{ux} & \ell_{uu} \end{bmatrix} [d x & d u]
\end{aligned}
$$

$$
+ V_{k+1} + V_{k+1,x}^T (A\,d x + B\,d u) + \tfrac12\ (A\,d x + B\,d u)^T V_{k+1,xx}\,(A\,d x + B\,d u).
$$

**记局部二次近似（以增量 $d x$, $d u$ 表达）**：

$$
Q(x + d x, u + d u)=\underbrace{c}_{\text{常数}} + Q_x^Td x + Q_u^Td u + \tfrac12d x^T Q_{xx} d x + d x^T Q_{xu} d u + \tfrac12 d u^T Q_{uu} d u.
$$


-- 一次项：
$$
Q_x = \ell_x + A^T V_{k+1,x} \\
Q_u = \ell_u + B^T V_{k+1,x}.
$$

-- 二次项（块矩阵）
$$
Q_{xx} = \ell_{xx} + A^T V_{k+1,xx} A \\
Q_{uu} = \ell_{uu} + B^T V_{k+1,xx} B \\
Q_{ux} = \ell_{ux} + B^T V_{k+1,xx} A.
$$


**最优增量控制**（二次子问题的解析解，带 Quu 正则）：
$$d u^* = k + K\,d x,\qquad k = -Q_{uu}^{-1}Q_u,\quad K = -Q_{uu}^{-1}Q_{ux}.$$

正则化以保证数值稳定:
$$
k = -\,\mathrm{inv}(Q_{uu}+\mu I)\;Q_u,
\qquad
K = -\,\mathrm{inv}(Q_{uu}+\mu I)\;Q_{ux}.
$$

**价值函数更新（递推假设）**：  
把最优控制代入Q得到新的状态值的一阶/二阶导数（常用、且经化简的形式）：
$$
V^{\text{new}}(x + d x) = c + Q_u^T k + \tfrac{1}{2} k^T Q_{uu} k + \big(Q_x + Q_{xu}^T k + Q_u^T K\big)^T d x + \tfrac{1}{2} d x^T \big(Q_{xx} + K^T Q_{uu} K + Q_{xu} K + K^T Q_{xu}^T\big) d x.
$$
或者是带入协态方程
$$
\lambda_k = l_x + l_{xu}\, d u_k + A_k^T \lambda_{k+1}
$$

1) **$V_x^{\text{new}}$ 的推导**：

$V_x^{\text{new}}$ 表示 $V^{\text{new}}$ 对状态 $x$ 的梯度，由 $V^{\text{new}}$ 的线性项系数给出：
$$
V_x^{\text{new}} = Q_x - Q_{ux}^T Q_{uu}^{-1} Q_u.
$$

2) **$V_{xx}^{\text{new}}$ 的推导**：

$V_{xx}^{\text{new}}$ 表示 $V^{\text{new}}$ 对状态 $x$ 的二阶导数，由 $V^{\text{new}}$ 的二次项系数给出：
$$
V_{xx}^{\text{new}} = Q_{xx} - Q_{ux}^T Q_{uu}^{-1} Q_{ux}.
$$


递推的核心是最优值函数的假设，或者哈密顿函数对应的协态方程  
所以，后续加入的不等式障碍函数项是不会影响递推逻辑的，也就是将多余的乘子项直接归入代价函数$l(x_k,u_k)$  
然后正常更新乘子、对新增的代价进行二阶泰勒展开即可  
$$
\hat{l}(x_k,u_k)=l(x_k,u_k)+...
$$

需要求解以下优化问题：
$$
\min \quad J = \sum_{k=0}^{N-1} L(x_k, u_k) + \Phi(x_N),
$$
约束条件为：
1. **动态约束**：
$$
x_{k+1} =f(x_k, u_k), 
$$
2. **不等式约束**：
$$
g(x_k, u_k) \leq 0.
$$
3. **等式约束**：
$$
h(x_k, u_k) = 0.
$$

# 障碍函数内点法
动态约束和等式约束需要分开，动态约束是递推必要的，等式约束不是，除非跨时间步(需要另外讨论)  
实际中，求解器没有等式约束的设置，只有不等式约束，也就是两个不等式约束当做一个等式约束，即：  
$$x=1$$
$$1 \leq x \leq 1$$
因此，这里就讨论不等式约束即可  

$$
\hat{l}(x_k,u_k)=l(x_k,u_k) -\mu \log(-g(x_k, y_k))
$$
所有时间步的障碍参数是共享的，在每次前向传播成功后，缩小障碍参数:  
$$
\mu \leftarrow \beta \mu, \quad \beta \in (0, 1).
$$
依旧是求取代价函数的各阶导数值，继续ILQR流程即可,值得注意的是，初解必须满足不等式约束，$g < 0$（严格内点，即约束内）。  
公式：$b(g) = -\log(-c)$
当 $g \to 0^-$ 时，$b(g) \to +\infty$，强烈惩罚接近边界；当 $g \to -\infty$ 时，$b(g) \to 0$。  
这是内点法的基础，确保优化器始终在可行域内。理论上收敛性强，适合精确约束优化（如线性规划）

## 指数障碍函数法

适用于 $g \leq 0$ 时接近0，$g > 0$ 时指数增长。  
允许轻微违反约束（$g > 0$），代价平滑增长；参数 $q_1$ 和 $q_2$ 可调。
在指数障碍函数法中，障碍函数的形式为：
$$
\hat{l}(x_k, u_k) = l(x_k, u_k) + \frac{1}{q_2} \exp(q_1 \cdot g(x_k, u_k))
$$
其中，$q_1$ 和 $q_2$ 是可调参数，控制惩罚的强度和形状。

### 参数更新机制
- **障碍参数更新**：在每次前向传播（forward pass）成功后（即轨迹代价下降），缩小障碍参数以增加对约束违反的惩罚强度：
  $$
  q_1 \leftarrow \beta \cdot q_1, \quad \beta \in (0, 1)
  $$
  这里，$\beta$ 是一个小于1的因子（如0.1到0.9），用于逐步减小参数，使惩罚逐渐增强。

- **为什么更新？**
  - 初始时，$q_1$ 较大，惩罚较弱，允许优化器探索可行域附近。
  - 随着迭代，减小 $q_1$，使指数项对约束违反更敏感，推动解向严格可行域收敛。
  - $q_2$ 可以保持固定或根据需要调整，以控制惩罚的幅度。

- 注意：初解必须满足不等式约束 $g(x_k, u_k) \leq 0$，以避免初始惩罚过大。

# Log  vs. Exponential 

## 为什么CILQR用指数而不是log？

### 允许近似/软约束：

Log barrier要求严格内点（$g < 0$），否则未定义或需要额外处理。在实时运动规划中，车辆可能短暂违反约束（如速度略超限），指数barrier更宽容，避免硬性失败。  
CILQR用于轨迹优化，指数barrier提供“软惩罚”，让优化器在边界附近更灵活，而非突然停止。

### 数值稳定性和平滑性：

Log barrier在 $g = 0$ 时有奇点（无穷大），可能导致数值不稳定或梯度爆炸。指数barrier更平滑，导数连续，适合梯度-based优化（如iLQR的backward pass）。  
在高维/非线性问题中，指数barrier收敛更快，且易于参数调优（$q_1$ 控制陡峭度，$q_2$ 控制幅度）。

# 障碍函数外点法
把约束违规作为惩罚项加到目标函数中，从而允许迭代点位于可行域之外，逐步“罚”回可行域。它和内点法的障碍函数，要求初解严格可行,正好相对。
$$
\hat{l}(x_k,u_k)=l(x_k,u_k) +\mu \max(0,g(x_k,u_k))^2
$$
迭代过程中检查约束违例，若违例，增大罚因子$\mu$  
随着罚因子增大问题会病态（条件数差），纯罚法可能收敛很慢或数值不稳定；增广拉格朗日通常比纯罚法更稳健。

# 增广拉格朗日法
增广拉格朗日法是在原始目标上同时引入拉格朗日乘子项与二次罚项，既利用乘子提供一阶约束信息，又避免纯罚法对罚因子过度依赖，从而通常更稳健地逼近 KKT 条件。  
首先构造等价问题:
$$
\hat{l}(x_k,u_k)=l(x_k,u_k) +\frac{\rho}{2} \max(0,g(x_k,u_k))^2
$$
再加入乘子法:
$$
\hat{l}(x_k,u_k)=l(x_k,u_k) +\frac{\rho}{2} \max(0,g(x_k,u_k)+\frac{\mu_k}{\rho})^2+\mu_k g(x_k,u_k)
$$
更新不等式约束的拉格朗日乘子 $\mu$：
$$
\mu_k^{(k+1)} = \max\left(0, \mu_k^{(k)} + \rho g(x_k, y_k)\right).
$$
增大惩罚参数 $\rho$：
$$
\rho^{(k+1)} = \gamma \rho^{(k)}, \quad \gamma > 1.
$$

## AL-ilqr迭代流程
### **初始化迭代参数**
$$
\rho \quad \vec{\mu} \quad \epsilon \quad \gamma
$$
#### **前向导数计算**
  - 首次运行时，需要初始状态$x_0$,同时需要初始控制序列$u_0->u_{N-1}$，作用是作为泰勒展开点
  - 前向模拟，从$x_0$出发，在每一时间点进行状态方程前向传递,记录状态序列$x_0->x_N$：
    $$x_{k+1}=f(x_k,u_k)$$
    - 同时计算代价函数各阶导数序列信息:
    $$
      l_{x,k} \quad l_{u,k} \quad l_{xu,k} \quad l_{uu,k} \quad A_{k} \quad B_{k}
    $$ 
#### **反向传播（Backward Pass）**

1. **初始条件（终端时刻 $N$）**：
   - 已知终端状态值函数 $V_N(x)$：
     - 常数项：$C_N$
     - 一阶项：$V_{N,x}$
     - 二阶项：$V_{N,xx}$
   - 即时代价函数 $l(x, u)$ 和系统动态 $f(x, u)$。

2. **从终端时刻 $N$ 反向递推到 $N-1$**：
   - **$Q$ 的各阶导数表达式**：
     - 一阶导数：
       $$
       Q_x = l_x + f_x^T V_{N,x}, \quad Q_u = l_u + f_u^T V_{N,x}.
       $$
     - 二阶导数：
       $$
       Q_{xx} = l_{xx} + f_x^T V_{N,xx} f_x, \quad Q_{uu} = l_{uu} + f_u^T V_{N,xx} f_u, \quad Q_{xu} = l_{xu} + f_x^T V_{N,xx} f_u.
       $$
     - 加入正则化，并且判断是否可逆(行列式是否为正)
      $$
      Q_{uu} = Q_{uu} + \mu I
      $$
      - if $|Q_{uu}|<1e-6$ break;
      - 如果不可逆，$\mu*=10$然后重新从N开始反向传播


   - 求解最优 $d u_{N-1}^*$：
     $$
     d u_{N-1}^* = k_{N-1} + K_{N-1} d x 
     $$
     不要显式求逆，使用线性方程解法(求解器会处理)
     $$
     Q_{uu}k_{N-1} = Q_u->k_{N-1} \\
     Q_{uu}K_{N-1} = -Q_{ux}->K_{N-1}.
     $$
     - **记录信息**：
       - 控制增量策略 $k_{N-1}$ 和 $K_{N-1}$。

   - 更新 $V_{N-1}$：
     $$
     \begin{aligned}
     C_{N-1} &= C_N - \tfrac{1}{2} Q_u^T Q_{uu}^{-1} Q_u=C_N+\tfrac{1}{2} Q_u^Tk_{N-1}, \\
     V_{N-1,x} &= Q_x - Q_{ux}^T Q_{uu}^{-1} Q_u=Q_x+Q_{ux}^Tk_{N-1}, \\
     V_{N-1,xx} &= Q_{xx} - Q_{ux}^T Q_{uu}^{-1} Q_{ux}=Q_{xx}+Q_{ux}^TK_{N-1}.
     \end{aligned}
     $$
     - 对称化保证：
      $$
      V_{xx}^{(k)} \leftarrow \tfrac12\big(V_{xx}^{(k)} + {V_{xx}^{(k)}}^T\big)
      $$
     - **记录信息**：
       - 更新后的值函数项 $C_{N-1}$、$V_{N-1,x}$ 和 $V_{N-1,xx}$。

3. **重复递推**：
   - 按上述步骤从 $N-1$ 递推到 $0$，依次计算 $V_{k,x}$ 和 $V_{k,xx}$。
   - **记录信息**：
     - 每个时刻的控制策略序列 $k_k$ 和 $K_k$。
     - 每个时刻的值函数项 $V_{k,x}$ 和 $V_{k,xx}$。
  
#### **前向传播（Forward Pass）**

1. **初始条件（起始时刻 $0$）**：
   - 已知初始状态 $x_0$。

2. **从 $x_0$ 前向传播到 $x_N$**：
   - 前向回放与线搜索（步长 $\alpha$）

     - 对每个候选 $\alpha$ 构造前向轨迹：
       $$\tilde u_k = u_k + \alpha\, k^{(k)} + K^{(k)} (\tilde x_k - x_k),\quad \tilde x_{k+1} = f(\tilde x_k,\tilde u_k)$$

     - 轨迹代价：
       $$J(\tilde X,\tilde U) = \sum_{k=0}^{N-1} \ell(\tilde x_k,\tilde u_k) + \Phi(\tilde x_N)$$

     - 接受条件：若 $J(\tilde X,\tilde U) < J_{\text{prev}}$，则接受并置 $J_{\text{prev}}\leftarrow J(\tilde X,\tilde U)$。

     - 若对所有 $\alpha$ 均不下降，则视为线搜索失败，执行：
       $$\mu \leftarrow \tau_{inc} \mu$$
       并重新进行后向传播。
       
#### **正则化自适应与收敛判据**

- 策略：若后向或线搜索失败，增大 $\mu$；若前向被接受，尝试减小：
  $$\mu \leftarrow \max(\mu_{\min},\;\mu/\tau_{dec})\quad(\tau_{dec}=10)$$

- 收敛判据（选用之一或组合）：
  $$|J_{\text{prev}}-J^{\text{new}}| < \varepsilon_J\quad\text{或}\quad \max_k \|k^{(k)}\|_{\infty} < \varepsilon_u\, .$$